# Actual Guardrails AI validation

Use this directory/shared root kernel. Six synthetic dictionaries are explicit response fixtures, not observed model output. Disable telemetry before importing the framework. No credentials or API calls are needed by default.

In [ ]:
import os
os.environ.update(DO_NOT_TRACK="1", OTEL_SDK_DISABLED="true", HF_HUB_OFFLINE="1", LITELLM_LOCAL_MODEL_COST_MAP="true")
import json
from pathlib import Path
from pprint import pprint
fixtures = json.loads(Path("docs/dataset/outputs.json").read_text())
pprint(fixtures)

## Read the native guard helper before importing

`tools/validation.py` builds actual Guard objects, disables metrics, and registers an ordinary validator function using the supported decorator. It returns native PassResult/FailResult, with a suggested masked fix. validation_view exposes actual schema errors and registered-validator history failures; it is not a substitute validator.

In [ ]:
from tools.validation import SCHEMA, contract_guard, email_guard, validation_view
contract = contract_guard()
pprint(SCHEMA)

## Contract fixtures, no re-ask

Pass serialized fixture objects to actual Guard.parse. Inspect native raw/validated output and errors. This SDK coerces numeric answer 20 into string 20; missing topic and invalid category still fail. Structure does not establish truth.

In [ ]:
contract_results = {}
for item in fixtures:
    outcome = contract.parse(json.dumps(item["raw"]), num_reasks=0)
    contract_results[item["id"]] = validation_view(outcome, contract)
pprint(contract_results)
assert contract_results["missing_field"]["validation_passed"] is False
assert contract_results["wrong_category"]["validation_passed"] is False
assert contract_results["wrong_type"]["validated"]["answer"] == "20"

## Withholding with a registered email validator

Start with on_fail=refrain and no model call/re-ask. We validate only the answer strings from the normal, email, and obfuscated fixtures. A detected email produces a real validator failure and withheld value; obfuscated wording is a visible miss.

In [ ]:
strings = {item["id"]: item["raw"]["answer"] for item in fixtures if item["id"] in ["normal", "email", "obfuscated"]}
refrain = email_guard("refrain")
refrain_results = {}
for name, text in strings.items():
    outcome = refrain.parse(text, num_reasks=0)
    refrain_results[name] = validation_view(outcome, refrain)
pprint(refrain_results)
assert refrain_results["email"]["exposed"] is None

## Change only on_fail to FIX, same strings

The final FIX outcome can pass because it repaired the text. The original registered-rule failure stays visible; it was not a clean original pass. This narrow regex still misses the obfuscated address.

In [ ]:
fix = email_guard("fix")
fix_results = {}
for name, text in strings.items():
    outcome = fix.parse(text, num_reasks=0)
    fix_results[name] = validation_view(outcome, fix)
    pprint({"id": name, "refrain": refrain_results[name], "fix": fix_results[name]})
assert fix_results["email"]["exposed"] == "Contact [EMAIL]."
assert fix_results["email"]["original_validator_failures"]
assert fix_results["obfuscated"]["validation_passed"] is True

## One bounded re-ask through a local fixture callback

Only this step permits one re-ask. The SDK parameter is named llm_api, but the supplied callback is ordinary local fixture code. It actually logs each invocation. One scenario repairs the schema; another is still invalid after the limit and remains withheld.

In [ ]:
reask_logs, reask_results = {}, {}
initial_invalid = '{"answer":20}'
for label, reply in [("recovers", '{"topic":"leave","answer":"20 days"}'), ("still_invalid", '{"topic":"wrong","answer":"20 days"}')]:
    callback_log = []
    def fixture_callback(*args, messages=None, **kwargs):
        callback_log.append({"messages": messages, "source": "local fixture callback"})
        return reply
    guard = contract_guard()
    outcome = guard.parse(initial_invalid, llm_api=fixture_callback, num_reasks=1, messages=[{"role": "user", "content": "Return the required JSON object."}], full_schema_reask=True)
    reask_logs[label] = callback_log
    reask_results[label] = validation_view(outcome, guard)
    pprint({"scenario": label, "initial_invalid_fixture": initial_invalid, "actual_local_callback_calls": len(callback_log), "api_calls": 0, "result": reask_results[label]})
assert all(len(log) == 1 for log in reask_logs.values())
assert reask_results["recovers"]["validation_passed"] is True
assert reask_results["still_invalid"]["exposed"] is None

## Optional actual model output — disabled

Read models/groq_model.py and prompts/answer.txt before enabling. The loader reads the ignored root .env only during the live function call; default offline execution never reads it. The raw provider text then goes through the same actual Guard.parse with zero re-asks. No live result was obtained during preparation.

In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    from models.groq_model import generate_live
    raw_live = generate_live("How much annual leave do full-time employees get?", model_name="openai/gpt-oss-20b")
    live_guard = contract_guard()
    live_outcome = live_guard.parse(raw_live, num_reasks=0)
    pprint({"provenance": "actual Groq model output", "validation": validation_view(live_outcome, live_guard)})
else:
    print("Live step skipped: no model/API calls or credential reads.")